# 04. Модель прогноза и ошибка

Цель ноутбука: обучить табличную модель прогноза спроса на подготовленных признаках и сравнить качество с baseline.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.config import PROCESSED_DATA_DIR, RESULTS_DIR
from src.metrics import metrics_table
from src.modeling import predict_non_negative, train_regression_model
from src.validation import make_time_series_folds, split_by_fold

In [ ]:
features = pd.read_parquet(PROCESSED_DATA_DIR / 'features.parquet')
features['date'] = pd.to_datetime(features['date'])

feature_columns = [
    'avg_unit_price',
    'invoices',
    'customers',
    'day_of_week',
    'week_of_year',
    'month',
    'is_weekend',
    'lag_7',
    'lag_14',
    'lag_28',
    'rolling_mean_7',
    'rolling_mean_28',
]

model_frame = features.dropna(subset=feature_columns + ['sales']).copy()
fold = make_time_series_folds(model_frame['date'], validation_size=28, n_folds=1)[0]
train, valid = split_by_fold(model_frame, fold)

In [ ]:
model = train_regression_model(train, feature_columns)
valid = valid.copy()
valid['forecast'] = predict_non_negative(model, valid, feature_columns)

model_metrics = metrics_table(valid['sales'], valid['forecast'])
model_metrics

In [ ]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
valid[['date', 'stock_code', 'country', 'sales', 'forecast']].to_parquet(RESULTS_DIR / 'forecast_validation.parquet', index=False)
model_metrics.to_csv(RESULTS_DIR / 'model_metrics.csv', index=False)
valid.head()

## Вывод после запуска

- WMAPE модели: `[A]`;
- MAE модели: `[B]`;
- RMSE модели: `[C]`;
- forecast bias: `[D]`.